# 02B — Pemodelan LSTM pada Data Baru (v2): 3 Skenario Simulasi Ketimpangan
Notebook ini menguji performa model LSTM unidirectional pada 3 skenario rasio ketimpangan data latih:
1. **Skenario 1:1:1** (Seimbang Sempurna: 1.087 tiap kelas, total 3.261 sampel)
2. **Skenario 6:3:1** (Ketimpangan Moderat: 3.372 Neg, 1.686 Pos, 562 Net)
3. **Skenario 8:1:1** (Ketimpangan Ekstrem / Long-tail: 3.376 Neg, 422 Pos, 422 Net)

Seluruh model dievaluasi pada data uji empiris terkunci ($n = 1.730$).


In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, accuracy_score, f1_score

if Path.cwd().name == 'notebooks':
    os.chdir('..')

tf.random.set_seed(42)
np.random.seed(42)

# Load data uji test set
df_full = pd.read_csv('Data/processed/banjir_processed_v2.csv')
from sklearn.model_selection import train_test_split
_, test_df = train_test_split(df_full, test_size=0.20, stratify=df_full['label'], random_state=42)
y_test = test_df['label'].values
print(f'Data Uji Test Set: {len(test_df)} sampel')


Data Uji Test Set: 1730 sampel


In [2]:
def build_lstm_sim():
    model = Sequential([
        Embedding(10000, 128),
        LSTM(64),
        Dropout(0.2),
        Dense(3, activation='softmax')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

print('Arsitektur LSTM siap untuk simulasi.')


Arsitektur LSTM siap untuk simulasi.


## Rangkuman Temuan Simulasi LSTM
| Skenario | Accuracy | Macro F1 | Recall Netral | Diagnosa Ilmiah |
| :--- | :---: | :---: | :---: | :--- |
| **1:1:1** | 42,54% | 42,14% | **77,48%** | Sensitivitas minoritas tinggi, namun akurasi global turun drastis |
| **6:3:1** | 71,85% | 51,42% | 0,00% | Terjadi **Majority Collapse** (Netral diabaikan) |
| **8:1:1** | 54,16% | 23,42% | 0,00% | Keruntuhan parah pada ketimpangan ekstrem |
